# 1 · CNN from Scratch — Cats vs Dogs

A convolutional neural network built and trained **from zero** on a small cats-vs-dogs dataset. This is the baseline you compare every transfer-learning model against.

**Steps:** download data → load → visualise → build CNN → train → plot curves → confusion matrix.

> Tip: turn on the GPU — *Runtime → Change runtime type → GPU*.

In [1]:
# Run this ONCE to fix TensorFlow + Keras compatibility
!pip install --force-reinstall tensorflow==2.17.0 tensorflow-intel==2.17.0 --quiet
!pip install keras==3.4.1 scikit-learn matplotlib --quiet


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.12.3 requires wrapt<2.0.0,>=1.10.10, but you have wrapt 2.2.2 which is incompatible.
astroid 2.14.2 requires wrapt<2,>=1.14; python_version >= "3.11", but you have wrapt 2.2.2 which is incompatible.
composio-core 0.7.21 requires Pillow<11,>=10.2.0, but you have pillow 12.1.1 which is incompatible.
composio-core 0.7.21 requires rich<14,>=13.7.1, but you have rich 15.0.0 which is incompatible.
deprecated 1.2.18 requires wrapt<2,>=1.10, but you have wrapt 2.2.2 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.16.3 which is incompatible.
google-genai 1.28.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
google-genai 1.28.0 requires websockets<15.1.0,>=13.0.0, but you have websockets 12.0 which is incompatible.
gradio 5.22.0 requires 

### Step 1 — Imports

In [2]:
# Standard imports
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print("TensorFlow version:", tf.__version__)
print("GPU detected      :", tf.config.list_physical_devices("GPU"))
# If the list above is empty, enable the GPU:
# Runtime -> Change runtime type -> Hardware accelerator -> GPU


ModuleNotFoundError: No module named 'tensorflow'

### Step 2 — Set up dataset paths

In [ ]:
# Point to the local full dog-vs-cat dataset
# (already extracted under ./data/dog-cat-full-dataset/)
base_dir  = os.path.join("data", "dog-cat-full-dataset", "data")
train_dir = os.path.join(base_dir, "train")   # ~10 000 cats + ~10 000 dogs
val_dir   = os.path.join(base_dir, "test")     # 2 500 cats + 2 500 dogs

for split in [train_dir, val_dir]:
    for cls in sorted(os.listdir(split)):
        n = len(os.listdir(os.path.join(split, cls)))
        print(f"{split}/{cls}: {n} images")


### Step 3 — Load images into `tf.data` datasets

In [ ]:
# Load images straight from the folders. Sub-folder names become the labels.
IMG_SIZE   = (160, 160)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=123,
)

# shuffle=False on validation keeps the order fixed so the confusion
# matrix labels line up with the predictions later.
val_ds = keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)


### Step 4 — Look at the data

In [ ]:
# Peek at a few training images
plt.figure(figsize=(9, 9))
for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.suptitle("Sample training images")
plt.tight_layout()
plt.show()


### Step 5 — Speed-up pipeline (cache + prefetch)

In [ ]:
# For a from-scratch CNN we normalise inside the model (a Rescaling layer),
# so here we only cache + prefetch for speed.
AUTOTUNE   = tf.data.AUTOTUNE
train_prep = train_ds.cache().prefetch(AUTOTUNE)
val_prep   = val_ds.cache().prefetch(AUTOTUNE)


### Step 6 — Build the CNN

In [ ]:
# A small CNN built from scratch: 4 conv/pool blocks + a dense head.
# Rescaling(1/255) normalises pixels; data augmentation fights overfitting.
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

model = keras.Sequential([
    keras.Input(shape=IMG_SIZE + (3,)),
    data_augmentation,
    layers.Rescaling(1.0 / 255),

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
], name="cnn_from_scratch")

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()


### Step 7 — Train

In [ ]:
EPOCHS = 15
history = model.fit(train_prep, validation_data=val_prep, epochs=EPOCHS)


### Step 8 — Accuracy & loss curves

In [ ]:
# Training vs validation curves — the quickest way to spot over/under-fitting
acc      = history.history["accuracy"]
val_acc  = history.history["val_accuracy"]
loss     = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc,     marker="o", label="Train")
plt.plot(epochs_range, val_acc, marker="o", label="Validation")
plt.title("Accuracy"); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss,     marker="o", label="Train")
plt.plot(epochs_range, val_loss, marker="o", label="Validation")
plt.title("Loss"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Step 9 — Confusion matrix & report

In [ ]:
# Evaluate + confusion matrix + per-class report
val_loss_v, val_acc_v = model.evaluate(val_prep, verbose=0)
print(f"Validation accuracy: {val_acc_v:.4f}")

# True labels (val_ds was NOT shuffled, so the order is stable)
y_true = np.concatenate([y.numpy() for _, y in val_ds], axis=0)

# Predicted labels
y_prob = model.predict(val_prep)
y_pred = np.argmax(y_prob, axis=1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
plt.grid(False)
plt.show()

print("\nClassification report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


### Step 10 — See predictions

In [ ]:
# Visualise predictions on a batch of validation images
# (the model already contains a Rescaling layer, so we predict on raw images)
plt.figure(figsize=(12, 12))
for images, labels in val_ds.take(1):
    probs = model.predict(images, verbose=0)
    preds = np.argmax(probs, axis=1)
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        true_lbl = class_names[labels[i]]
        pred_lbl = class_names[preds[i]]
        color = "green" if true_lbl == pred_lbl else "red"
        plt.title(f"pred: {pred_lbl}\ntrue: {true_lbl}", color=color, fontsize=10)
        plt.axis("off")
plt.tight_layout()
plt.show()


---
**Takeaway:** a from-scratch CNN on only 2000 images tends to overfit and plateaus around ~70–80% val accuracy. The transfer-learning notebooks reach ~95%+ with far less training — that's the whole point.